In [ ]:
!pip install -q transformers accelerate bitsandbytes peft trl datasets kagglehub scikit-learn pynvml

In [ ]:
import os
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from PIL import Image
from tqdm import tqdm
from datasets import Dataset

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_curve,
    auc
)

from transformers import (
    AutoProcessor,
    LlavaForConditionalGeneration,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)

device = "cuda" if torch.cuda.is_available() else "cpu"

print(device)

In [ ]:
import kagglehub

dataset_path = kagglehub.dataset_download(
    "parthplc/facebook-hateful-meme-dataset"
)

print(dataset_path)

In [ ]:
#Load Dataset Paths

TRAIN_FILE = os.path.join(
    dataset_path,
    "data/train.jsonl"
)

DEV_FILE = os.path.join(
    dataset_path,
    "data/dev.jsonl"
)

TEST_FILE = os.path.join(
    dataset_path,
    "data/test.jsonl"
)

IMAGE_DIR = os.path.join(
    dataset_path,
    "data"
)

print(TRAIN_FILE)
print(DEV_FILE)
print(TEST_FILE)

In [ ]:
# ==========================================
# LOAD ONLY FIRST 500 SAMPLES
# ==========================================

def load_jsonl(file_path, limit=None):

    data = []

    with open(file_path, "r") as f:

        for idx, line in enumerate(f):

            if limit and idx >= limit:
                break

            sample = json.loads(line)

            data.append(sample)

    return data


train_data = load_jsonl(
    TRAIN_FILE,
    limit=500
)

dev_data = load_jsonl(
    DEV_FILE,
    limit=100
)

test_data = load_jsonl(
    TEST_FILE,
    limit=100
)

print("TRAIN:", len(train_data))
print("DEV:", len(dev_data))
print("TEST:", len(test_data))

In [ ]:
sample = train_data[0]

image_path = os.path.join(
    IMAGE_DIR,
    sample["img"]
)

image = Image.open(image_path).convert("RGB")

plt.figure(figsize=(6,6))

plt.imshow(image)

plt.axis("off")

plt.show()

print(sample["text"])

In [ ]:
def convert_to_llava_format(data):

    converted = []

    for sample in data:

        image_path = os.path.join(
            IMAGE_DIR,
            sample["img"]
        )

        answer = (
            "Yes"
            if sample["label"] == 1
            else "No"
        )

        converted.append({

            "image": image_path,

            "prompt":
            f"""
USER:
<image>

Meme Text:
{sample["text"]}

Is this meme hateful?

ASSISTANT:
""",

            "answer": answer
        })

    return converted


train_converted = convert_to_llava_format(
    train_data
)

dev_converted = convert_to_llava_format(
    dev_data
)

In [ ]:
train_dataset = Dataset.from_list(
    train_converted
)

dev_dataset = Dataset.from_list(
    dev_converted
)

print(train_dataset)


In [ ]:
MODEL_ID = "llava-hf/llava-1.5-7b-hf"

bnb_config = BitsAndBytesConfig(

    load_in_4bit=True,

    bnb_4bit_quant_type="nf4",

    bnb_4bit_compute_dtype=torch.float16
)

processor = AutoProcessor.from_pretrained(
    MODEL_ID
)

model = LlavaForConditionalGeneration.from_pretrained(

    MODEL_ID,

    quantization_config=bnb_config,

    device_map="auto"
)

print("MODEL LOADED")

In [ ]:
model = prepare_model_for_kbit_training(
    model
)

lora_config = LoraConfig(

    r=16,

    lora_alpha=32,

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ],

    lora_dropout=0.05,

    bias="none",

    task_type="CAUSAL_LM"
)

model = get_peft_model(
    model,
    lora_config
)

model.print_trainable_parameters()

In [ ]:
def preprocess_function(example):

    image = Image.open(
        example["image"]
    ).convert("RGB")

    prompt = (
        example["prompt"] +
        example["answer"]
    )

    inputs = processor(

        text=prompt,

        images=image,

        return_tensors="pt",

        padding=True,

        truncation=False
    )

    inputs = {

        k: v.squeeze(0)

        for k, v in inputs.items()
    }

    inputs["labels"] = inputs[
        "input_ids"
    ].clone()

    return inputs


train_dataset = train_dataset.map(
    preprocess_function
)

dev_dataset = dev_dataset.map(
    preprocess_function
)

print("PREPROCESSING COMPLETED")

In [ ]:
training_args = TrainingArguments(

    output_dir="./llava_output",

    per_device_train_batch_size=1,

    gradient_accumulation_steps=1,

    num_train_epochs=10,

    learning_rate=2e-4,

    fp16=True,

    logging_steps=5,

    save_strategy="no",

    report_to="none"
)

In [ ]:
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=dev_dataset
)

In [ ]:
trainer.train()

In [ ]:
def predict_llava(image_path, text):

    image = Image.open(
        image_path
    ).convert("RGB")

    prompt = f"""
USER:
<image>

Meme Text:
{text}

Is this meme hateful?

ASSISTANT:
"""

    inputs = processor(

        text=prompt,

        images=image,

        return_tensors="pt"
    ).to(device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=10
        )

    result = processor.batch_decode(
        outputs,
        skip_special_tokens=True
    )[0]

    result = result.lower()

    if "yes" in result:
        return 1
    else:
        return 0

In [ ]:
GPU_AVAILABLE = torch.cuda.is_available()

if GPU_AVAILABLE:

    try:

        from pynvml import *

        nvmlInit()

        handle = nvmlDeviceGetHandleByIndex(0)

        ENERGY_MODE = "GPU"

    except:

        ENERGY_MODE = "CPU"

else:

    ENERGY_MODE = "CPU"


def get_power_usage():

    if ENERGY_MODE == "GPU":

        return (
            nvmlDeviceGetPowerUsage(handle)
            / 1000
        )

    else:

        return 65


def compute_energy_kwh(
    power_watts,
    duration_seconds
):

    return (
        power_watts *
        duration_seconds
    ) / (1000 * 3600)

In [ ]:
y_true = []

y_pred = []

y_scores = []

inference_times = []

energy_consumptions = []

for sample in tqdm(dev_data[:100]):

    image_path = os.path.join(
        IMAGE_DIR,
        sample["img"]
    )

    text = sample["text"]

    label = sample["label"]

    start_time = time.time()

    power = get_power_usage()

    prediction = predict_llava(
        image_path,
        text
    )

    end_time = time.time()

    duration = end_time - start_time

    energy = compute_energy_kwh(
        power,
        duration
    )

    y_true.append(label)

    y_pred.append(prediction)

    y_scores.append(prediction)

    inference_times.append(duration)

    energy_consumptions.append(energy)

In [ ]:
# ============================================================
# FINAL EVALUATION
# ============================================================

from sklearn.metrics import (

    accuracy_score,

    precision_score,

    recall_score,

    f1_score,

    classification_report,

    confusion_matrix
)

accuracy = accuracy_score(

    y_true,

    y_pred
)

precision = precision_score(

    y_true,

    y_pred
)

recall = recall_score(

    y_true,

    y_pred
)

f1 = f1_score(

    y_true,

    y_pred
)

avg_time = np.mean(

    inference_times
)

avg_energy = np.mean(

    energy_consumptions
)

report = classification_report(

    y_true,

    y_pred,

    target_names=[

        "Non-Hateful",

        "Hateful"
    ],

    digits=4
)

print("Accuracy:", accuracy)

print("Precision:", precision)

print("Recall:", recall)

print("F1-score:", f1)

print("Inference Time:", avg_time)

print("Energy (kWh):", avg_energy)

print()

print("Classification Report:")

print(report)

In [ ]:
accuracy = accuracy_score(
    y_true,
    y_pred
)

precision = precision_score(
    y_true,
    y_pred
)

recall = recall_score(
    y_true,
    y_pred
)

f1 = f1_score(
    y_true,
    y_pred
)

avg_time = np.mean(
    inference_times
)

avg_energy = np.mean(
    energy_consumptions
)

print("Accuracy:", accuracy)

print("Precision:", precision)

print("Recall:", recall)

print("F1-score:", f1)

print("Inference Time:", avg_time)

print("Energy (kWh):", avg_energy)

In [ ]:
fpr, tpr, thresholds = roc_curve(
    y_true,
    y_scores
)

roc_auc = auc(
    fpr,
    tpr
)

plt.figure(figsize=(6,5))

plt.plot(
    fpr,
    tpr,
    label=f"AUC = {roc_auc:.4f}"
)

plt.plot(
    [0,1],
    [0,1],
    linestyle="--"
)

plt.xlabel("FPR")

plt.ylabel("TPR")

plt.title("ROC Curve")

plt.legend()

plt.show()

In [ ]:
results_df = pd.DataFrame({

    "Metric": [

        "Accuracy",

        "Precision",

        "Recall",

        "F1-score",

        "ROC-AUC",

        "Inference Time (s)",

        "Energy (kWh)"
    ],

    "Value": [

        accuracy,

        precision,

        recall,

        f1,

        roc_auc,

        avg_time,

        avg_energy
    ]
})

results_df.to_csv(
    "llava_results.csv",
    index=False
)

results_df

In [ ]:
model.save_pretrained(
    "llava_hateful_memes_lora"
)

processor.save_pretrained(
    "llava_hateful_memes_lora"
)

print("MODEL SAVED")